# SAS ↔ Python — Notebook de référence comparative
## Formation Beobank · Orsys

Ce notebook regroupe **toutes** les correspondances SAS → Python utilisées pendant la formation.
Il ne fait pas partie du fil rouge des exercices : consultez-le à tout moment, en dehors des notebooks Jour 1/2/3
qui restent concentrés sur Python.

**Sommaire**
1. Variables et types
2. Valeurs manquantes
3. Conditions
4. Boucles
5. Formats / tables de correspondance
6. Exploration de données (PROC PRINT / CONTENTS / MEANS / FREQ)
7. Tri et filtre (PROC SORT / WHERE)
8. Colonne calculée (IF/THEN/ELSE vs np.where / np.select)
9. PROC SQL vs pandas / sqlite3
10. Export (PROC EXPORT vs to_csv)
11. Aide-mémoire récapitulatif

## 1. Variables et types

In [ ]:
# --- SAS ---
# DATA _NULL_;
#    idt_ac = "AC00001";
#    solde  = 15234.50;
#    PUT idt_ac= solde=;
# RUN;

# --- PYTHON --- (pas de déclaration de type : affectation directe)
idt_ac = "AC00001"     # str
solde  = 15234.50      # float
print(idt_ac, solde)

## 2. Valeurs manquantes

En SAS, `.` (numérique) et `""` (texte) sont automatiquement traités comme manquants par la plupart
des PROC. En Python, rien n'est automatique : il faut gérer les manquants explicitement.

In [ ]:
# --- SAS ---
# IF solde = . THEN solde = 0;              /* numérique manquant */
# PROC MEANS DATA=ctr NMISS; VAR solde; RUN; /* compter les manquants */

# --- PYTHON --- (valeur seule : None ; dans pandas : NaN / pd.NA)
def convertir_solde_robuste(valeur):
    try:
        return float(valeur)
    except (ValueError, TypeError):
        return None       # équivalent Python du "." SAS

print(convertir_solde_robuste("15234.50"))   # 15234.5
print(convertir_solde_robuste("."))          # None

# Avec pandas : isna() / fillna() / dropna()
import pandas as pd
serie = pd.Series([15234.50, None, 8900.0])
print(serie.isna().sum())        # 1 manquant
print(serie.fillna(0).tolist())  # [15234.5, 0.0, 8900.0]

## 3. Conditions

In [ ]:
# --- SAS ---
# IF cod = "1" THEN libelle = "Ouvert";
# ELSE IF cod = "2" THEN libelle = "En attente";
# ELSE libelle = "Inconnu";

# --- PYTHON --- (indentation obligatoire, elif = "else if")
cod = "2"
if cod == "1":
    libelle = "Ouvert"
elif cod == "2":
    libelle = "En attente"
else:
    libelle = "Inconnu"
print(libelle)

## 4. Boucles

In [ ]:
# --- SAS ---
# DO i = 1 TO 12;
#    PUT "Mois " i;
# END;

# --- PYTHON --- (range(1, 13) : 1 inclus, 13 EXCLU)
for i in range(1, 13):
    print("Mois", i)

## 5. Formats / tables de correspondance

In [ ]:
# --- SAS ---
# PROC FORMAT;
#    VALUE $STATUT "1"="Ouvert" "2"="En attente" "3"="Suspendu";
# RUN;
# DATA ctr; SET ctr; libelle = PUT(cod, $STATUT.); RUN;

# --- PYTHON --- (dict + .map(), ou .get() cellule par cellule)
statuts = {"1": "Ouvert", "2": "En attente", "3": "Suspendu"}

cod = "2"
print(statuts.get(cod, "Inconnu"))

import pandas as pd
ctr_demo = pd.DataFrame({"COD_ECV_CTR": ["1", "2", "3"]})
ctr_demo["LIBELLE"] = ctr_demo["COD_ECV_CTR"].map(statuts)
print(ctr_demo)

## 6. Exploration de données

In [ ]:
import pandas as pd
ctr = pd.read_csv("../data/CTR.csv", sep=";", na_values=".")

# --- SAS ---                              --- PYTHON ---
# PROC PRINT DATA=ctr(OBS=5); RUN;         ctr.head()
# PROC CONTENTS DATA=ctr; RUN;             ctr.info()
# PROC MEANS DATA=ctr; VAR SLD_CTR; RUN;   ctr["SLD_CTR"].describe()
# PROC FREQ DATA=ctr;
#    TABLES COD_ECV_CTR; RUN;              ctr["COD_ECV_CTR"].value_counts()

print(ctr.head())
print(ctr["COD_ECV_CTR"].value_counts())

## 7. Tri et filtre

In [ ]:
# --- SAS ---
# PROC SORT DATA=ctr; BY SLD_CTR; RUN;
# DATA clotures; SET ctr; WHERE COD_ECV_CTR = 4; RUN;

# --- PYTHON ---
# COD_ECV_CTR est un entier dans le fichier réel : on compare à 4 (pas "4")
ctr_trie    = ctr.sort_values("SLD_CTR")
ctr_cloture = ctr[ctr["COD_ECV_CTR"] == 4]          # filtre booléen = WHERE
ctr_inactif = ctr[ctr["COD_ECV_CTR"].isin([4, 6])]  # .isin() = IN()

print(len(ctr_trie), len(ctr_cloture), len(ctr_inactif))

## 8. Colonne calculée

In [ ]:
import numpy as np

# --- SAS ---
# DATA ctr; SET ctr;
#    IF SLD_CTR > 0 THEN FLAG = "P"; ELSE FLAG = "N";
# RUN;

# --- PYTHON --- (np.where = 1 condition, np.select = plusieurs)
ctr["FLAG"] = np.where(ctr["SLD_CTR"] > 0, "P", "N")

ctr["SEGMENT"] = np.select(
    [ctr["SLD_CTR"] < 0, ctr["SLD_CTR"] < 5000],
    ["Critique", "Faible"],
    default="Élevé"
)
print(ctr[["SLD_CTR", "FLAG", "SEGMENT"]].head())

## 9. PROC SQL vs pandas / sqlite3

In [ ]:
# --- SAS ---
# PROC SQL;
#    SELECT COD_ECV_CTR, COUNT(*) AS N
#    FROM ctr
#    GROUP BY COD_ECV_CTR
#    HAVING N > 10;
# QUIT;

# --- PYTHON (pandas) ---
resultat_pandas = (ctr.groupby("COD_ECV_CTR")
                       .size()
                       .reset_index(name="N")
                       .query("N > 10"))
print(resultat_pandas)

# --- PYTHON (SQL réel via sqlite3) --- même résultat, syntaxe SQL identique à PROC SQL
import sqlite3
conn = sqlite3.connect(":memory:")
ctr.to_sql("ctr", conn, index=False)
requete_sql = "SELECT COD_ECV_CTR, COUNT(*) AS N FROM ctr GROUP BY COD_ECV_CTR HAVING N > 10"
resultat_sql = pd.read_sql(requete_sql, conn)
print(resultat_sql)

## 10. Export

In [ ]:
# --- SAS ---
# PROC EXPORT DATA=rapport OUTFILE="rapport.csv" DBMS=DLM DELIMITER=";"; RUN;

# --- PYTHON ---
ctr.head(10).to_csv("../output/rapport.csv", sep=";", index=False)
print("Export écrit : ../output/rapport.csv")

## 11. Aide-mémoire récapitulatif

| SAS | Python |
|---|---|
| `DATA step` | boucle `for` + `if` sur des dicts, ou colonne pandas |
| `.` (manquant numérique) | `None` / `NaN` |
| `IF/THEN/ELSE IF/ELSE` | `if/elif/else` |
| `DO i = 1 TO n; END;` | `for i in range(1, n+1):` |
| `PROC FORMAT` + `PUT(var, fmt.)` | `dict` + `.map()` |
| `PROC PRINT(OBS=n)` | `.head(n)` |
| `PROC CONTENTS` | `.info()` |
| `PROC MEANS` | `.describe()` |
| `PROC FREQ` | `.value_counts()` |
| `PROC SORT BY var` | `.sort_values("var")` |
| `WHERE cond` | `df[cond]` |
| `IN (...)` | `.isin([...])` |
| `IF cond THEN a ELSE b` (colonne) | `np.where(cond, a, b)` |
| `SELECT/WHEN/OTHERWISE` (colonne) | `np.select([...], [...], default=...)` |
| `PROC SQL` | `pandas` (groupby/merge) ou `sqlite3` avec le même SQL |
| `PROC EXPORT` | `.to_csv()` |
| `%MACRO / %MEND` | `def ... / return` |